# PAA: auditoría y dataset preliminar de viabilidad

Este notebook audita FIRMS e INUMET y construye un panel preliminar `departamento-semana`. FIRMS representa **detecciones de anomalías térmicas/focos de calor**, no incendios confirmados. No se entrenan modelos ni se sobrescriben los archivos originales.

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 1. Carga y control inicial

In [ ]:
firms = pd.read_parquet(
    "/home/pepo/PAA/data/firms_2018_2025.parquet"
)

inumet = pd.read_parquet(
    "/home/pepo/PAA/data/inumet_2018_2025.parquet"
)

departamentos_geo = gpd.read_file(
    "/home/pepo/PAA/geoBoundaries-URY-ADM1-all/"
    "geoBoundaries-URY-ADM1.geojson"
)[["shapeName", "shapeISO", "geometry"]]

pd.DataFrame({
    "dataset": ["FIRMS", "INUMET", "Departamentos"],
    "filas": [len(firms), len(inumet), len(departamentos_geo)],
    "columnas": [firms.shape[1], inumet.shape[1], departamentos_geo.shape[1]]
})

In [ ]:
pd.DataFrame({
    "nulos": firms.isna().sum(),
    "dtype": firms.dtypes.astype(str)
}).query("nulos > 0"), firms.duplicated().sum()

## 2. Selección de Uruguay y coherencia temporal

In [ ]:
datos_uruguay = firms.loc[firms["pais"].eq("URY")].copy()

# FIRMS declara acq_date/acq_time en UTC. utc=True etiqueta esos valores como UTC;
# no debe usarse para convertir una hora local que no haya sido normalizada antes.
datos_uruguay["fecha_hora_utc"] = pd.to_datetime(
    datos_uruguay["fecha_adq"].astype(str)
    + " "
    + datos_uruguay["hora_adq_hhmm"].astype(str).str.zfill(4),
    format="%Y-%m-%d %H%M",
    utc=True
)

auditoria_temporal = pd.DataFrame({
    "fuente": ["FIRMS", "INUMET"],
    "dtype": [str(datos_uruguay["fecha_hora_utc"].dtype), str(inumet["fecha_hora_utc"].dtype)],
    "zona_horaria": [str(datos_uruguay["fecha_hora_utc"].dt.tz), str(inumet["fecha_hora_utc"].dt.tz)],
    "fecha_minima": [datos_uruguay["fecha_hora_utc"].min(), inumet["fecha_hora_utc"].min()],
    "fecha_maxima": [datos_uruguay["fecha_hora_utc"].max(), inumet["fecha_hora_utc"].max()]
})

auditoria_temporal

In [ ]:
# Control de plausibilidad, no prueba definitiva de la conversión original de INUMET.
temperatura_por_hora_utc = (
    inumet.assign(hora_utc=inumet["fecha_hora_utc"].dt.hour)
    .groupby("hora_utc", as_index=False)
    .agg(temperatura_media_c=("temperatura_c", "mean"))
)
temperatura_por_hora_utc.round(2)

## 3. Departamentos y revisión de puntos no asignados

In [ ]:
puntos_uruguay = gpd.GeoDataFrame(
    datos_uruguay,
    geometry=gpd.points_from_xy(datos_uruguay["longitud"], datos_uruguay["latitud"]),
    crs="EPSG:4326"
)

datos_uruguay_departamentos = gpd.sjoin(
    puntos_uruguay,
    departamentos_geo,
    how="left",
    predicate="within"
).rename(columns={"shapeName": "departamento"})

datos_uruguay_departamentos["departamento"].value_counts(dropna=False)

In [ ]:
sin_departamento = datos_uruguay_departamentos.loc[
    datos_uruguay_departamentos["departamento"].isna(),
    ["latitud", "longitud", "fecha_adq", "geometry"]
].copy()

departamentos_metros = departamentos_geo.to_crs("EPSG:32721")
sin_departamento_metros = sin_departamento.to_crs("EPSG:32721")
revision = []

for _, punto in sin_departamento_metros.iterrows():
    distancias_limite = departamentos_metros.geometry.distance(punto.geometry)
    indice_cercano = distancias_limite.idxmin()
    revision.append({
        "latitud": punto["latitud"],
        "longitud": punto["longitud"],
        "fecha": punto["fecha_adq"],
        "departamento_mas_cercano": departamentos_metros.loc[indice_cercano, "shapeName"],
        "distancia_limite_km": distancias_limite.loc[indice_cercano] / 1000
    })

revision_sin_departamento = pd.DataFrame(revision)
revision_sin_departamento.round({"latitud": 4, "longitud": 4, "distancia_limite_km": 3})

Los siete casos se conservan sin reasignar. Se excluyen solamente del panel departamental hasta una revisión cartográfica manual; no se eliminan de la fuente FIRMS.

## 4. Cobertura temporal real de INUMET

In [ ]:
cobertura_inumet_anual = (
    inumet.assign(anio=inumet["fecha_hora_utc"].dt.year)
    .groupby("anio", as_index=False)
    .agg(
        filas=("fecha_hora_utc", "size"),
        estaciones=("ubicacion", "nunique"),
        inicio=("fecha_hora_utc", "min"),
        fin=("fecha_hora_utc", "max"),
        temperatura_valida=("temperatura_c", "count"),
        humedad_valida=("humedad_pct", "count")
    )
)
cobertura_inumet_anual

In [ ]:
cobertura_estacion_anio = pd.crosstab(
    inumet["ubicacion"],
    inumet["fecha_hora_utc"].dt.year
)
cobertura_estacion_anio

## 5. Estación INUMET más cercana y cobertura espacial

In [ ]:
estaciones = (
    inumet[["ubicacion", "departamento", "latitud", "longitud"]]
    .drop_duplicates("ubicacion")
    .reset_index(drop=True)
)

def haversine_vectorizada(lat1, lon1, lat2, lon2):
    radio_tierra_km = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    diferencia_latitud = lat2 - lat1
    diferencia_longitud = lon2 - lon1
    a = (
        np.sin(diferencia_latitud / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(diferencia_longitud / 2) ** 2
    )
    return 2 * radio_tierra_km * np.arcsin(np.sqrt(a))

matriz_distancias = haversine_vectorizada(
    datos_uruguay_departamentos["latitud"].to_numpy()[:, None],
    datos_uruguay_departamentos["longitud"].to_numpy()[:, None],
    estaciones["latitud"].to_numpy()[None, :],
    estaciones["longitud"].to_numpy()[None, :]
)
indice_estacion = matriz_distancias.argmin(axis=1)
datos_uruguay_departamentos["estacion_inumet"] = estaciones.iloc[indice_estacion]["ubicacion"].to_numpy()
datos_uruguay_departamentos["distancia_inumet_km"] = matriz_distancias[np.arange(len(matriz_distancias)), indice_estacion]

In [ ]:
distancias = datos_uruguay_departamentos["distancia_inumet_km"]
resumen_distancias = pd.DataFrame({
    "metrica": ["mínimo", "media", "mediana", "p75", "p90", "p95", "p99", "máximo"],
    "distancia_km": [
        distancias.min(), distancias.mean(), distancias.median(),
        distancias.quantile(.75), distancias.quantile(.90),
        distancias.quantile(.95), distancias.quantile(.99), distancias.max()
    ]
})

umbrales = [25, 50, 75, 100, 150, 200]
cobertura_por_distancia = pd.DataFrame({
    "umbral_km": umbrales,
    "cantidad": [distancias.le(limite).sum() for limite in umbrales],
    "porcentaje": [distancias.le(limite).mean() * 100 for limite in umbrales]
})

resumen_distancias.round(2), cobertura_por_distancia.round(2)

## 6. Auditoría del merge temporal: `nearest` frente a `backward`

In [ ]:
firmas_merge = datos_uruguay_departamentos.copy()
firmas_merge["estacion_inumet"] = firmas_merge["estacion_inumet"].astype("string")

inumet_merge = inumet.rename(columns={
    "ubicacion": "estacion_inumet",
    "departamento": "departamento_inumet",
    "latitud": "latitud_inumet",
    "longitud": "longitud_inumet",
    "fecha_hora_utc": "fecha_inumet_utc"
}).copy()
inumet_merge["estacion_inumet"] = inumet_merge["estacion_inumet"].astype("string")

datos_inumet_nearest_auditoria = pd.merge_asof(
    firmas_merge.sort_values("fecha_hora_utc"),
    inumet_merge.sort_values("fecha_inumet_utc"),
    left_on="fecha_hora_utc", right_on="fecha_inumet_utc",
    by="estacion_inumet", direction="nearest",
    tolerance=pd.Timedelta("1 hour")
)

datos_inumet_backward_auditoria = pd.merge_asof(
    firmas_merge.sort_values("fecha_hora_utc"),
    inumet_merge.sort_values("fecha_inumet_utc"),
    left_on="fecha_hora_utc", right_on="fecha_inumet_utc",
    by="estacion_inumet", direction="backward",
    tolerance=pd.Timedelta("1 hour")
)

comparacion_merge = pd.DataFrame({
    "version": ["nearest", "backward"],
    "con_temperatura": [
        datos_inumet_nearest_auditoria["temperatura_c"].notna().sum(),
        datos_inumet_backward_auditoria["temperatura_c"].notna().sum()
    ],
    "con_humedad": [
        datos_inumet_nearest_auditoria["humedad_pct"].notna().sum(),
        datos_inumet_backward_auditoria["humedad_pct"].notna().sum()
    ]
})
comparacion_merge

`nearest` puede seleccionar una medición posterior a la detección. Para predicción debe preferirse información cerrada antes del objetivo; la meteorología final se agregará por semana previa y no por cada foco.

In [ ]:
datos_inumet_nearest_auditoria["anio"] = datos_inumet_nearest_auditoria["fecha_hora_utc"].dt.year
cobertura_merge_anual = (
    datos_inumet_nearest_auditoria.groupby("anio", as_index=False)
    .agg(
        total=("fecha_hora_utc", "size"),
        con_temperatura=("temperatura_c", "count"),
        con_humedad=("humedad_pct", "count")
    )
)
cobertura_merge_anual["pct_temperatura"] = cobertura_merge_anual["con_temperatura"] / cobertura_merge_anual["total"] * 100
cobertura_merge_anual["pct_humedad"] = cobertura_merge_anual["con_humedad"] / cobertura_merge_anual["total"] * 100

resumen_por_estacion = (
    datos_inumet_nearest_auditoria.groupby("estacion_inumet", as_index=False)
    .agg(
        detecciones=("fecha_hora_utc", "size"),
        con_temperatura=("temperatura_c", "count"),
        distancia_media_km=("distancia_inumet_km", "mean"),
        distancia_mediana_km=("distancia_inumet_km", "median"),
        distancia_p90_km=("distancia_inumet_km", lambda x: x.quantile(.90)),
        distancia_maxima_km=("distancia_inumet_km", "max")
    )
)

cobertura_merge_anual.round(2), resumen_por_estacion.round(2)

## 7. Representación departamental de las estaciones

In [ ]:
centroides = departamentos_geo.to_crs("EPSG:32721").copy()
centroides["geometry"] = centroides.geometry.centroid
centroides = centroides.to_crs("EPSG:4326")

distancias_centroides = haversine_vectorizada(
    centroides.geometry.y.to_numpy()[:, None],
    centroides.geometry.x.to_numpy()[:, None],
    estaciones["latitud"].to_numpy()[None, :],
    estaciones["longitud"].to_numpy()[None, :]
)
indice_centroide = distancias_centroides.argmin(axis=1)

representacion_departamental = pd.DataFrame({
    "departamento": centroides["shapeName"],
    "estacion_centroide": estaciones.iloc[indice_centroide]["ubicacion"].to_numpy(),
    "distancia_centroide_km": distancias_centroides[np.arange(len(centroides)), indice_centroide]
}).sort_values(["estacion_centroide", "distancia_centroide_km"])

representacion_departamental.round(2)

## 8. Panel preliminar departamento-semana

In [ ]:
detecciones_asignadas = datos_uruguay_departamentos.loc[
    datos_uruguay_departamentos["departamento"].notna()
].copy()

# Semana consistente con inicio en lunes, conservando UTC.
detecciones_asignadas["semana"] = (
    detecciones_asignadas["fecha_hora_utc"].dt.normalize()
    - pd.to_timedelta(detecciones_asignadas["fecha_hora_utc"].dt.weekday, unit="D")
)

conteos_semanales = (
    detecciones_asignadas.groupby(["departamento", "semana"])
    .size().rename("cantidad_focos")
)

nombres_departamentos = sorted(departamentos_geo["shapeName"].unique())
semanas = pd.date_range(
    detecciones_asignadas["semana"].min(),
    detecciones_asignadas["semana"].max(),
    freq="W-MON", tz="UTC"
)
indice_completo = pd.MultiIndex.from_product(
    [nombres_departamentos, semanas], names=["departamento", "semana"]
)

panel_departamento_semana_preliminar = (
    conteos_semanales.reindex(indice_completo, fill_value=0).reset_index()
)
panel_departamento_semana_preliminar["objetivo"] = (
    panel_departamento_semana_preliminar["cantidad_focos"].gt(0).astype("int8")
)

panel_departamento_semana_preliminar.head()

In [ ]:
total = len(panel_departamento_semana_preliminar)
positivos = int(panel_departamento_semana_preliminar["objetivo"].sum())
negativos = total - positivos

resumen_viabilidad = pd.DataFrame({
    "observaciones": [total],
    "positivos": [positivos],
    "negativos": [negativos],
    "porcentaje_positivo": [positivos / total * 100],
    "ratio_negativos_positivos": [negativos / positivos]
})

resumen_anual = (
    panel_departamento_semana_preliminar.assign(anio=lambda df: df["semana"].dt.year)
    .groupby("anio", as_index=False)
    .agg(observaciones=("objetivo", "size"), positivos=("objetivo", "sum"))
)
resumen_anual["porcentaje_positivo"] = resumen_anual["positivos"] / resumen_anual["observaciones"] * 100

resumen_departamental = (
    panel_departamento_semana_preliminar.groupby("departamento", as_index=False)
    .agg(observaciones=("objetivo", "size"), positivos=("objetivo", "sum"))
)
resumen_departamental["porcentaje_positivo"] = resumen_departamental["positivos"] / resumen_departamental["observaciones"] * 100

resumen_viabilidad.round(2), resumen_anual.round(2), resumen_departamental.sort_values("positivos", ascending=False).round(2)

## 9. FIRMS completo frente a semanas con cobertura potencial INUMET

In [ ]:
inumet_semanal = (
    inumet.assign(
        semana=inumet["fecha_hora_utc"].dt.normalize()
        - pd.to_timedelta(inumet["fecha_hora_utc"].dt.weekday, unit="D")
    )
    .groupby("semana", as_index=False)
    .agg(filas_inumet=("fecha_hora_utc", "size"), estaciones_disponibles=("ubicacion", "nunique"))
)

panel_periodo_inumet = panel_departamento_semana_preliminar.loc[
    panel_departamento_semana_preliminar["semana"].isin(inumet_semanal["semana"])
].copy()

def resumir_panel(panel, nombre):
    total_panel = len(panel)
    positivos_panel = int(panel["objetivo"].sum())
    return {
        "panel": nombre,
        "semanas": panel["semana"].nunique(),
        "observaciones": total_panel,
        "positivos": positivos_panel,
        "negativos": total_panel - positivos_panel,
        "porcentaje_positivo": positivos_panel / total_panel * 100
    }

comparacion_paneles = pd.DataFrame([
    resumir_panel(panel_departamento_semana_preliminar, "FIRMS completo"),
    resumir_panel(panel_periodo_inumet, "Semanas con alguna observación INUMET")
])

semanas_por_numero_estaciones = (
    inumet_semanal["estaciones_disponibles"].value_counts().sort_index()
    .rename_axis("estaciones_disponibles").rename("semanas").reset_index()
)

comparacion_paneles.round(2), semanas_por_numero_estaciones

## 10. Conclusión y próximo paso

El panel es útil para evaluar viabilidad, pero todavía no es el dataset final de entrenamiento. La red INUMET disponible tiene huecos completos en 2022–2023 y representación espacial débil para varios departamentos.

El próximo paso recomendado es auditar `meteo_2018_2025.parquet` y `chirps_2018_2025.parquet`, construir variables meteorológicas agregadas por departamento y semana anterior, incorporar rezagos de focos y crear `objetivo_t_plus_1`. La última semana debe excluirse al no disponer de una etiqueta futura completa. No deben utilizarse brillo, FRP ni confianza FIRMS como predictores.